# Лабораторная работа 8.1  
## Управление секретами и доступами

Тема: безопасное управление секретами и доступами в инфраструктуре ML-платформы.

В ноутбуке собирается единый проект, содержащий:

- пример FastAPI-приложения;
- Dockerfile;
- GitHub Actions workflow;
- HashiCorp Vault policies;
- скрипт настройки Vault;
- скрипт проверки Vault-доступов;
- Kubernetes namespace;
- Kubernetes ServiceAccount;
- Kubernetes Role;
- Kubernetes RoleBinding;
- скрипт проверки Kubernetes RBAC;
- README;
- шаблон отчёта REPORT.md.

> Важно: все секреты в проекте являются демонстрационными.  
> Реальные production-секреты, пароли, токены и ключи использовать запрещено.

In [ ]:
from pathlib import Path

# Создаём структуру каталогов лабораторной работы.
directories = [
    ".github/workflows",
    "vault/policies",
    "k8s",
    "app",
    "reports",
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)

print("Структура проекта создана.")

## 1. Демонстрационное приложение

Минимальное FastAPI-приложение используется как пример компонента ML-платформы.

Оно не выводит секреты в логи и только проверяет, что необходимые переменные окружения переданы.

In [ ]:
Path("app/main.py").write_text(r'''
import os
from fastapi import FastAPI

app = FastAPI(
    title="Demo ML Platform App",
    description="Demo service for secrets and RBAC lab",
    version="1.0.0",
)


@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "demo-ml-platform-app",
    }


@app.get("/config-check")
def config_check():
    # Никогда не возвращаем значения секретов.
    # Возвращаем только факт наличия переменных окружения.
    required_vars = [
        "MLFLOW_TRACKING_URI",
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
    ]

    return {
        "required_variables_present": {
            name: bool(os.getenv(name))
            for name in required_vars
        }
    }
'''.strip() + "\n", encoding="utf-8")

Path("app/requirements.txt").write_text(r'''
fastapi==0.115.6
uvicorn[standard]==0.34.0
'''.strip() + "\n", encoding="utf-8")

Path("app/Dockerfile").write_text(r'''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir --upgrade pip \
    && pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''.strip() + "\n", encoding="utf-8")

print("Файлы демонстрационного приложения созданы.")

## 2. Git ignore

Файл `.gitignore` защищает от случайной публикации локальных файлов окружения, токенов и kubeconfig.

In [ ]:
Path(".gitignore").write_text(r'''
# Python
__pycache__/
*.pyc
.venv/
venv/
env/

# Jupyter
.ipynb_checkpoints/

# Secrets and local env files
.env
.env.*
*.key
*.pem
*.crt
kubeconfig
kubeconfig.yaml
vault-token.txt

# OS
.DS_Store

# Reports artifacts
reports/*.png
reports/*.csv
'''.strip() + "\n", encoding="utf-8")

print(".gitignore создан.")

## 3. Vault policies

Создаются политики доступа HashiCorp Vault для ролей:

- `data-scientist`;
- `ml-engineer`;
- `ci-cd`;
- `mlflow`;
- `model-serving`;
- `admin`.

Используется KV Secrets Engine v2, поэтому в policies путь к данным имеет вид:

```text
secret/data/ml-platform/...
```

In [ ]:
Path("vault/policies/data-scientist-policy.hcl").write_text(r'''
# Data Scientist может читать параметры MLflow и read-only доступ к S3.
# Он не должен иметь доступ к production DB, registry write tokens и serving secrets.

path "secret/data/ml-platform/mlflow" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/s3-readonly" {
  capabilities = ["read"]
}

path "secret/metadata/ml-platform/*" {
  capabilities = ["list"]
}
'''.strip() + "\n", encoding="utf-8")

Path("vault/policies/ml-engineer-policy.hcl").write_text(r'''
# ML Engineer может читать секреты, необходимые для пайплайнов обучения и деплоя.

path "secret/data/ml-platform/mlflow" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/s3-readwrite" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/model-registry" {
  capabilities = ["read"]
}

path "secret/metadata/ml-platform/*" {
  capabilities = ["list"]
}
'''.strip() + "\n", encoding="utf-8")

Path("vault/policies/ci-cd-policy.hcl").write_text(r'''
# CI/CD получает только секреты, необходимые для сборки, тестирования и деплоя.
# Полный admin-доступ запрещён.

path "secret/data/ml-platform/mlflow" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/s3-readwrite" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/model-registry" {
  capabilities = ["read"]
}
'''.strip() + "\n", encoding="utf-8")

Path("vault/policies/mlflow-policy.hcl").write_text(r'''
# MLflow Tracking Server читает пароль БД и S3-доступ для хранения артефактов.

path "secret/data/ml-platform/postgres" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/s3-readwrite" {
  capabilities = ["read"]
}
'''.strip() + "\n", encoding="utf-8")

Path("vault/policies/model-serving-policy.hcl").write_text(r'''
# Model Serving читает только секреты, необходимые для inference.
# Доступ к PostgreSQL и секретам обучения запрещён.

path "secret/data/ml-platform/model-serving" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/model-registry" {
  capabilities = ["read"]
}

path "secret/data/ml-platform/s3-readonly" {
  capabilities = ["read"]
}
'''.strip() + "\n", encoding="utf-8")

Path("vault/policies/admin-policy.hcl").write_text(r'''
# Admin имеет полный доступ только к зоне ml-platform.
# В реальном production доступ admin должен быть ограничен организационными процедурами.

path "secret/data/ml-platform/*" {
  capabilities = ["create", "read", "update", "delete", "list"]
}

path "secret/metadata/ml-platform/*" {
  capabilities = ["create", "read", "update", "delete", "list"]
}
'''.strip() + "\n", encoding="utf-8")

print("Vault policies созданы.")

## 4. Скрипт настройки Vault

Скрипт `vault/setup-vault.sh`:

1. проверяет соединение с Vault;
2. включает KV v2 engine;
3. создаёт демонстрационные секреты;
4. загружает policies;
5. создаёт demo-токены для проверки.

Перед запуском Vault можно поднять локально:

```bash
docker run --cap-add=IPC_LOCK \
  -e VAULT_DEV_ROOT_TOKEN_ID=root \
  -e VAULT_DEV_LISTEN_ADDRESS=0.0.0.0:8200 \
  -p 8200:8200 \
  hashicorp/vault:1.15
```

Затем:

```bash
export VAULT_ADDR=http://127.0.0.1:8200
export VAULT_TOKEN=root
bash vault/setup-vault.sh
```

In [ ]:
Path("vault/setup-vault.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

# Скрипт настройки демонстрационного Vault для лабораторной работы.
# Все секреты являются учебными и не должны использоваться в production.

: "${VAULT_ADDR:=http://127.0.0.1:8200}"
: "${VAULT_TOKEN:=root}"

export VAULT_ADDR
export VAULT_TOKEN

echo "Vault address: ${VAULT_ADDR}"
echo "Checking Vault status..."
vault status || {
  echo "Vault is not available. Start Vault first."
  exit 1
}

echo "Enabling KV v2 secrets engine at path secret/ if needed..."
vault secrets enable -path=secret kv-v2 2>/dev/null || true

echo "Writing demo secrets..."

vault kv put secret/ml-platform/mlflow \
  MLFLOW_TRACKING_URI="http://mlflow.ml-platform.local" \
  MLFLOW_TRACKING_USERNAME="ml_user" \
  MLFLOW_TRACKING_PASSWORD="example-password"

vault kv put secret/ml-platform/s3-readonly \
  AWS_ACCESS_KEY_ID="demo-readonly-access-key" \
  AWS_SECRET_ACCESS_KEY="demo-readonly-secret-key" \
  AWS_DEFAULT_REGION="us-east-1" \
  S3_ENDPOINT_URL="http://minio.ml-platform.local"

vault kv put secret/ml-platform/s3-readwrite \
  AWS_ACCESS_KEY_ID="demo-readwrite-access-key" \
  AWS_SECRET_ACCESS_KEY="demo-readwrite-secret-key" \
  AWS_DEFAULT_REGION="us-east-1" \
  S3_ENDPOINT_URL="http://minio.ml-platform.local"

vault kv put secret/ml-platform/postgres \
  POSTGRES_HOST="postgres.ml-platform.local" \
  POSTGRES_DB="mlflow" \
  POSTGRES_USER="mlflow" \
  POSTGRES_PASSWORD="example-postgres-password"

vault kv put secret/ml-platform/model-registry \
  REGISTRY_URL="registry.example.com" \
  REGISTRY_USERNAME="registry-user" \
  REGISTRY_PASSWORD="example-registry-password"

vault kv put secret/ml-platform/model-serving \
  MODEL_SERVING_API_TOKEN="example-serving-token" \
  MODEL_BUCKET="models" \
  MODEL_PATH="models/production/model.pkl"

echo "Writing Vault policies..."

vault policy write data-scientist-policy vault/policies/data-scientist-policy.hcl
vault policy write ml-engineer-policy vault/policies/ml-engineer-policy.hcl
vault policy write ci-cd-policy vault/policies/ci-cd-policy.hcl
vault policy write mlflow-policy vault/policies/mlflow-policy.hcl
vault policy write model-serving-policy vault/policies/model-serving-policy.hcl
vault policy write admin-policy vault/policies/admin-policy.hcl

echo "Creating demo tokens..."
mkdir -p vault/tokens

vault token create -policy=data-scientist-policy -ttl=1h -format=json | jq -r '.auth.client_token' > vault/tokens/data-scientist.token
vault token create -policy=ml-engineer-policy -ttl=1h -format=json | jq -r '.auth.client_token' > vault/tokens/ml-engineer.token
vault token create -policy=ci-cd-policy -ttl=1h -format=json | jq -r '.auth.client_token' > vault/tokens/ci-cd.token
vault token create -policy=mlflow-policy -ttl=1h -format=json | jq -r '.auth.client_token' > vault/tokens/mlflow.token
vault token create -policy=model-serving-policy -ttl=1h -format=json | jq -r '.auth.client_token' > vault/tokens/model-serving.token
vault token create -policy=admin-policy -ttl=1h -format=json | jq -r '.auth.client_token' > vault/tokens/admin.token

echo "Vault setup completed."
echo "Demo tokens are stored in vault/tokens/ and ignored by git."
'''.strip() + "\n", encoding="utf-8")

print("vault/setup-vault.sh создан.")

## 5. Скрипт проверки Vault-доступов

Скрипт проверяет положительные и отрицательные сценарии:

- `model-serving` может читать serving-секрет;
- `model-serving` не может читать PostgreSQL-секрет;
- `data-scientist` может читать MLflow;
- `data-scientist` не может читать S3 read/write;
- `ci-cd` не имеет административного доступа;
- `admin` имеет полный доступ в зоне `ml-platform`.

In [ ]:
Path("vault/test-vault-access.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

: "${VAULT_ADDR:=http://127.0.0.1:8200}"
export VAULT_ADDR

TOKENS_DIR="vault/tokens"

function check_allowed() {
  local token_file="$1"
  local command="$2"
  local description="$3"

  echo ""
  echo "[ALLOW EXPECTED] ${description}"

  VAULT_TOKEN="$(cat "${TOKENS_DIR}/${token_file}")" bash -c "${command}" >/dev/null
  echo "OK: allowed"
}

function check_denied() {
  local token_file="$1"
  local command="$2"
  local description="$3"

  echo ""
  echo "[DENY EXPECTED] ${description}"

  set +e
  VAULT_TOKEN="$(cat "${TOKENS_DIR}/${token_file}")" bash -c "${command}" >/dev/null 2>&1
  status=$?
  set -e

  if [ "$status" -eq 0 ]; then
    echo "ERROR: access was allowed, but denial was expected"
    exit 1
  else
    echo "OK: denied"
  fi
}

check_allowed "model-serving.token" "vault kv get secret/ml-platform/model-serving" "model-serving reads model-serving secret"
check_denied  "model-serving.token" "vault kv get secret/ml-platform/postgres" "model-serving cannot read postgres secret"

check_allowed "data-scientist.token" "vault kv get secret/ml-platform/mlflow" "data-scientist reads MLflow secret"
check_denied  "data-scientist.token" "vault kv get secret/ml-platform/s3-readwrite" "data-scientist cannot read S3 read/write secret"

check_allowed "ci-cd.token" "vault kv get secret/ml-platform/model-registry" "ci-cd reads model registry secret"
check_denied  "ci-cd.token" "vault kv delete secret/ml-platform/mlflow" "ci-cd cannot delete secrets"

check_allowed "admin.token" "vault kv get secret/ml-platform/postgres" "admin reads postgres secret"

echo ""
echo "Vault access tests completed."
'''.strip() + "\n", encoding="utf-8")

print("vault/test-vault-access.sh создан.")

## 6. GitHub Actions workflow

Workflow показывает безопасный шаблон:

- параметры подключения к Vault берутся из GitHub Secrets;
- runtime-секреты читаются из Vault;
- значения секретов не печатаются в логах;
- выполняется только проверка наличия переменных.

In [ ]:
Path(".github/workflows/ml-ci.yml").write_text(r'''
name: ML Platform CI

on:
  push:
    branches:
      - main
  pull_request:

jobs:
  ci:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Read secrets from Vault
        uses: hashicorp/vault-action@v3
        with:
          url: ${{ secrets.VAULT_ADDR }}
          token: ${{ secrets.VAULT_TOKEN }}
          secrets: |
            secret/data/ml-platform/mlflow MLFLOW_TRACKING_URI | MLFLOW_TRACKING_URI ;
            secret/data/ml-platform/s3-readwrite AWS_ACCESS_KEY_ID | AWS_ACCESS_KEY_ID ;
            secret/data/ml-platform/s3-readwrite AWS_SECRET_ACCESS_KEY | AWS_SECRET_ACCESS_KEY ;

      - name: Check required environment variables
        run: |
          echo "Checking required variables"
          test -n "$MLFLOW_TRACKING_URI"
          test -n "$AWS_ACCESS_KEY_ID"
          test -n "$AWS_SECRET_ACCESS_KEY"
          echo "Secrets are available, values are not printed"

      - name: Run simple security check
        run: |
          echo "Checking that forbidden files are not committed"
          test ! -f .env
          test ! -f vault-token.txt
'''.strip() + "\n", encoding="utf-8")

print(".github/workflows/ml-ci.yml создан.")

## 7. Kubernetes RBAC

Создаются:

- namespace `ml-platform`;
- service accounts;
- роли;
- role bindings;
- демонстрационные secrets/configmaps;
- скрипт проверки доступов.

Проверка выполняется командой:

```bash
bash k8s/test-rbac.sh
```

In [ ]:
Path("k8s/namespace.yaml").write_text(r'''
apiVersion: v1
kind: Namespace
metadata:
  name: ml-platform
'''.strip() + "\n", encoding="utf-8")

Path("k8s/service-accounts.yaml").write_text(r'''
apiVersion: v1
kind: ServiceAccount
metadata:
  name: mlflow-sa
  namespace: ml-platform
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: training-job-sa
  namespace: ml-platform
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: model-serving-sa
  namespace: ml-platform
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: ci-cd-deployer-sa
  namespace: ml-platform
---
apiVersion: v1
kind: ServiceAccount
metadata:
  name: data-scientist-sa
  namespace: ml-platform
'''.strip() + "\n", encoding="utf-8")

Path("k8s/demo-config-and-secrets.yaml").write_text(r'''
apiVersion: v1
kind: ConfigMap
metadata:
  name: ml-platform-config
  namespace: ml-platform
data:
  MLFLOW_TRACKING_URI: "http://mlflow.ml-platform.local"
  S3_ENDPOINT_URL: "http://minio.ml-platform.local"
---
apiVersion: v1
kind: Secret
metadata:
  name: model-serving-secret
  namespace: ml-platform
type: Opaque
stringData:
  MODEL_SERVING_API_TOKEN: "demo-serving-token"
---
apiVersion: v1
kind: Secret
metadata:
  name: mlflow-db-secret
  namespace: ml-platform
type: Opaque
stringData:
  POSTGRES_PASSWORD: "demo-postgres-password"
---
apiVersion: v1
kind: Secret
metadata:
  name: s3-readwrite-secret
  namespace: ml-platform
type: Opaque
stringData:
  AWS_ACCESS_KEY_ID: "demo-readwrite-access-key"
  AWS_SECRET_ACCESS_KEY: "demo-readwrite-secret-key"
'''.strip() + "\n", encoding="utf-8")

print("Kubernetes namespace, service accounts и demo secrets созданы.")

In [ ]:
Path("k8s/roles.yaml").write_text(r'''
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  namespace: ml-platform
  name: model-serving-role
rules:
  - apiGroups: [""]
    resources: ["pods", "services", "configmaps"]
    verbs: ["get", "list", "watch"]

  - apiGroups: [""]
    resources: ["secrets"]
    resourceNames: ["model-serving-secret"]
    verbs: ["get"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  namespace: ml-platform
  name: ci-cd-deployer-role
rules:
  - apiGroups: [""]
    resources: ["pods", "services", "configmaps"]
    verbs: ["get", "list", "watch", "create", "update", "patch"]

  - apiGroups: ["apps"]
    resources: ["deployments"]
    verbs: ["get", "list", "watch", "create", "update", "patch"]

  - apiGroups: ["batch"]
    resources: ["jobs"]
    verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  namespace: ml-platform
  name: training-job-role
rules:
  - apiGroups: [""]
    resources: ["pods", "configmaps"]
    verbs: ["get", "list", "watch", "create"]

  - apiGroups: ["batch"]
    resources: ["jobs"]
    verbs: ["get", "list", "watch", "create"]

  - apiGroups: [""]
    resources: ["secrets"]
    resourceNames: ["s3-readwrite-secret"]
    verbs: ["get"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  namespace: ml-platform
  name: mlflow-role
rules:
  - apiGroups: [""]
    resources: ["pods", "services", "configmaps"]
    verbs: ["get", "list", "watch"]

  - apiGroups: [""]
    resources: ["secrets"]
    resourceNames: ["mlflow-db-secret", "s3-readwrite-secret"]
    verbs: ["get"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  namespace: ml-platform
  name: data-scientist-role
rules:
  - apiGroups: [""]
    resources: ["pods", "pods/log", "configmaps"]
    verbs: ["get", "list", "watch"]

  - apiGroups: ["batch"]
    resources: ["jobs"]
    verbs: ["get", "list", "watch"]
'''.strip() + "\n", encoding="utf-8")

Path("k8s/role-bindings.yaml").write_text(r'''
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: model-serving-binding
  namespace: ml-platform
subjects:
  - kind: ServiceAccount
    name: model-serving-sa
    namespace: ml-platform
roleRef:
  kind: Role
  name: model-serving-role
  apiGroup: rbac.authorization.k8s.io
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: ci-cd-deployer-binding
  namespace: ml-platform
subjects:
  - kind: ServiceAccount
    name: ci-cd-deployer-sa
    namespace: ml-platform
roleRef:
  kind: Role
  name: ci-cd-deployer-role
  apiGroup: rbac.authorization.k8s.io
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: training-job-binding
  namespace: ml-platform
subjects:
  - kind: ServiceAccount
    name: training-job-sa
    namespace: ml-platform
roleRef:
  kind: Role
  name: training-job-role
  apiGroup: rbac.authorization.k8s.io
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: mlflow-binding
  namespace: ml-platform
subjects:
  - kind: ServiceAccount
    name: mlflow-sa
    namespace: ml-platform
roleRef:
  kind: Role
  name: mlflow-role
  apiGroup: rbac.authorization.k8s.io
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: data-scientist-binding
  namespace: ml-platform
subjects:
  - kind: ServiceAccount
    name: data-scientist-sa
    namespace: ml-platform
roleRef:
  kind: Role
  name: data-scientist-role
  apiGroup: rbac.authorization.k8s.io
'''.strip() + "\n", encoding="utf-8")

print("Kubernetes roles и role-bindings созданы.")

In [ ]:
Path("k8s/test-rbac.sh").write_text(r'''
#!/usr/bin/env bash
set -euo pipefail

NS="ml-platform"

function expect_yes() {
  local command="$1"
  local description="$2"

  echo ""
  echo "[YES EXPECTED] ${description}"

  result=$(eval "${command}")
  echo "Result: ${result}"

  if [ "${result}" != "yes" ]; then
    echo "ERROR: expected yes"
    exit 1
  fi
}

function expect_no() {
  local command="$1"
  local description="$2"

  echo ""
  echo "[NO EXPECTED] ${description}"

  result=$(eval "${command}")
  echo "Result: ${result}"

  if [ "${result}" != "no" ]; then
    echo "ERROR: expected no"
    exit 1
  fi
}

echo "Applying Kubernetes RBAC manifests..."

kubectl apply -f k8s/namespace.yaml
kubectl apply -f k8s/service-accounts.yaml
kubectl apply -f k8s/demo-config-and-secrets.yaml
kubectl apply -f k8s/roles.yaml
kubectl apply -f k8s/role-bindings.yaml

echo ""
echo "Testing RBAC..."

expect_yes "kubectl auth can-i get secret/model-serving-secret --as=system:serviceaccount:${NS}:model-serving-sa -n ${NS}" \
  "model-serving-sa can read model-serving-secret"

expect_no "kubectl auth can-i get secret/mlflow-db-secret --as=system:serviceaccount:${NS}:model-serving-sa -n ${NS}" \
  "model-serving-sa cannot read mlflow-db-secret"

expect_yes "kubectl auth can-i create deployments --as=system:serviceaccount:${NS}:ci-cd-deployer-sa -n ${NS}" \
  "ci-cd-deployer-sa can create deployments"

expect_no "kubectl auth can-i delete secrets --as=system:serviceaccount:${NS}:ci-cd-deployer-sa -n ${NS}" \
  "ci-cd-deployer-sa cannot delete secrets"

expect_yes "kubectl auth can-i get pods/log --as=system:serviceaccount:${NS}:data-scientist-sa -n ${NS}" \
  "data-scientist-sa can read pod logs"

expect_no "kubectl auth can-i get secrets --as=system:serviceaccount:${NS}:data-scientist-sa -n ${NS}" \
  "data-scientist-sa cannot read secrets"

expect_yes "kubectl auth can-i create jobs --as=system:serviceaccount:${NS}:training-job-sa -n ${NS}" \
  "training-job-sa can create jobs"

expect_yes "kubectl auth can-i get secret/s3-readwrite-secret --as=system:serviceaccount:${NS}:training-job-sa -n ${NS}" \
  "training-job-sa can read s3-readwrite-secret"

echo ""
echo "RBAC tests completed successfully."
'''.strip() + "\n", encoding="utf-8")

print("k8s/test-rbac.sh создан.")

## 8. README и отчёт

Создадим README с инструкцией запуска и шаблон отчёта.

In [ ]:
Path("README.md").write_text(r'''
# Лабораторная работа 8.1
## Управление секретами и доступами

Проект демонстрирует безопасное управление секретами и доступами в учебной ML-платформе.

## Состав проекта

```text
.
├── .github/
│   └── workflows/
│       └── ml-ci.yml
├── vault/
│   ├── policies/
│   ├── setup-vault.sh
│   ├── test-vault-access.sh
│   └── README.md
├── k8s/
│   ├── namespace.yaml
│   ├── service-accounts.yaml
│   ├── demo-config-and-secrets.yaml
│   ├── roles.yaml
│   ├── role-bindings.yaml
│   └── test-rbac.sh
├── app/
│   ├── Dockerfile
│   ├── main.py
│   └── requirements.txt
├── REPORT.md
└── README.md
```

## 1. Запуск Vault

```bash
docker run --cap-add=IPC_LOCK \
  -e VAULT_DEV_ROOT_TOKEN_ID=root \
  -e VAULT_DEV_LISTEN_ADDRESS=0.0.0.0:8200 \
  -p 8200:8200 \
  hashicorp/vault:1.15
```

В другом терминале:

```bash
export VAULT_ADDR=http://127.0.0.1:8200
export VAULT_TOKEN=root
```

## 2. Настройка Vault

```bash
chmod +x vault/*.sh
bash vault/setup-vault.sh
```

Проверка доступов:

```bash
bash vault/test-vault-access.sh
```

## 3. GitHub Secrets

В GitHub Repository Secrets необходимо создать:

| Secret | Назначение |
|---|---|
| `VAULT_ADDR` | Адрес Vault |
| `VAULT_TOKEN` | Токен доступа к Vault |
| `KUBE_CONFIG` | kubeconfig для доступа к Kubernetes |
| `DOCKER_REGISTRY_USERNAME` | логин registry |
| `DOCKER_REGISTRY_PASSWORD` | пароль или token registry |

Важно: workflow не должен печатать значения секретов в логи.

## 4. Kubernetes RBAC

Применить RBAC и проверить доступы:

```bash
chmod +x k8s/test-rbac.sh
bash k8s/test-rbac.sh
```

Или вручную:

```bash
kubectl apply -f k8s/namespace.yaml
kubectl apply -f k8s/service-accounts.yaml
kubectl apply -f k8s/demo-config-and-secrets.yaml
kubectl apply -f k8s/roles.yaml
kubectl apply -f k8s/role-bindings.yaml
```

Проверка:

```bash
kubectl auth can-i get secret/model-serving-secret \
  --as=system:serviceaccount:ml-platform:model-serving-sa \
  -n ml-platform
```

## 5. Демонстрационное приложение

Локальный запуск:

```bash
cd app
pip install -r requirements.txt
uvicorn main:app --host 0.0.0.0 --port 8000
```

Проверка:

```bash
curl http://127.0.0.1:8000/health
curl http://127.0.0.1:8000/config-check
```

## 6. Основные принципы безопасности

- секреты не хранятся в Git;
- GitHub Secrets используются только для CI/CD bootstrap-параметров;
- Vault используется как централизованное хранилище секретов;
- роли получают только минимально необходимые права;
- проверяются как разрешённые, так и запрещённые действия;
- значения секретов не выводятся в логи.
```
'''.strip() + "\n", encoding="utf-8")

Path("vault/README.md").write_text(r'''
# Vault

Каталог содержит:

- политики доступа Vault;
- скрипт настройки Vault;
- скрипт проверки доступов.

## Запуск

```bash
export VAULT_ADDR=http://127.0.0.1:8200
export VAULT_TOKEN=root

bash vault/setup-vault.sh
bash vault/test-vault-access.sh
```
'''.strip() + "\n", encoding="utf-8")

Path("k8s/README.md").write_text(r'''
# Kubernetes RBAC

Каталог содержит:

- namespace;
- service accounts;
- demo secrets;
- roles;
- role bindings;
- скрипт проверки RBAC.

## Проверка

```bash
bash k8s/test-rbac.sh
```
'''.strip() + "\n", encoding="utf-8")

print("README-файлы созданы.")

In [ ]:
Path("REPORT.md").write_text(r'''
# Отчёт по лабораторной работе 8.1
## Управление секретами и доступами

## 1. Титульная информация

- Название лабораторной работы: Управление секретами и доступами
- ФИО студента:
- Группа:
- Дата выполнения:
- Ссылка на GitHub-репозиторий:

## 2. Описание архитектуры

Используемые компоненты:

- GitHub Repository;
- GitHub Actions;
- HashiCorp Vault;
- Kubernetes;
- Kubernetes RBAC;
- MLflow;
- S3/MinIO;
- Model Serving;
- Container Registry.

Схема:

```text
Developer
    |
    v
GitHub Repository
    |
    v
GitHub Actions
    |
    | uses GitHub Secrets
    v
HashiCorp Vault
    |
    | provides runtime secrets
    v
Kubernetes / ML Platform
    |
    +--> MLflow
    +--> Training Job
    +--> Model Serving
    +--> S3 / MinIO
```

## 3. Описание секретов

| Секрет | Где хранится | Кто имеет доступ | Назначение |
|---|---|---|---|
| MLflow credentials | Vault | data-scientist, ml-engineer, ci-cd | Работа с MLflow |
| S3 read-only key | Vault | data-scientist, model-serving | Чтение артефактов |
| S3 read-write key | Vault | training-job-sa, ci-cd | Запись моделей и артефактов |
| PostgreSQL password | Vault | mlflow-sa | Работа MLflow с БД |
| Registry token | GitHub Secrets / Vault | ci-cd | Push Docker image |
| Model serving token | Vault / Kubernetes Secret | model-serving-sa | Работа inference-сервиса |

## 4. Настройка Vault

Команды запуска Vault:

```bash
docker run --cap-add=IPC_LOCK \
  -e VAULT_DEV_ROOT_TOKEN_ID=root \
  -e VAULT_DEV_LISTEN_ADDRESS=0.0.0.0:8200 \
  -p 8200:8200 \
  hashicorp/vault:1.15
```

Настройка:

```bash
export VAULT_ADDR=http://127.0.0.1:8200
export VAULT_TOKEN=root
bash vault/setup-vault.sh
```

Проверка:

```bash
bash vault/test-vault-access.sh
```

Результаты проверки:

| Проверка | Ожидаемый результат | Фактический результат |
|---|---|---|
| model-serving читает model-serving secret | yes |  |
| model-serving читает postgres secret | no |  |
| data-scientist читает mlflow secret | yes |  |
| data-scientist читает s3-readwrite | no |  |
| ci-cd удаляет секреты | no |  |
| admin читает postgres secret | yes |  |

## 5. Настройка GitHub Secrets

Созданные GitHub Secrets:

| Secret | Назначение |
|---|---|
| VAULT_ADDR | Адрес Vault |
| VAULT_TOKEN | Токен доступа к Vault |
| KUBE_CONFIG | Доступ к Kubernetes |
| DOCKER_REGISTRY_USERNAME | Логин registry |
| DOCKER_REGISTRY_PASSWORD | Пароль/token registry |

Риски хранения `VAULT_TOKEN` в GitHub Secrets:

- токен может быть скомпрометирован при ошибке workflow;
- долгоживущий токен повышает риск несанкционированного доступа;
- сложнее обеспечить короткий TTL и ротацию.

Более безопасная альтернатива:

- GitHub OIDC;
- короткоживущие Vault tokens;
- ограниченные Vault policies;
- protected environments.

## 6. GitHub Actions workflow

Workflow:

- получает код;
- подключается к Vault;
- извлекает необходимые секреты;
- проверяет наличие переменных;
- не выводит значения секретов в логи.

Файл:

```text
.github/workflows/ml-ci.yml
```

## 7. Kubernetes RBAC

Namespace:

```text
ml-platform
```

Service Accounts:

- mlflow-sa;
- training-job-sa;
- model-serving-sa;
- ci-cd-deployer-sa;
- data-scientist-sa.

Таблица прав:

| Субъект | Pods | Jobs | Deployments | Services | ConfigMaps | Secrets |
|---|---|---|---|---|---|---|
| data-scientist-sa | get/list/logs | get/list | no | no | get/list | no |
| training-job-sa | get/list/create | get/list/create | no | no | get/list | only s3-readwrite-secret |
| model-serving-sa | get/list | no | no | get/list | get/list | only model-serving-secret |
| ci-cd-deployer-sa | get/list | get/list | create/update | create/update | create/update | no |
| mlflow-sa | get/list | no | no | get/list | get/list | mlflow-db-secret, s3-readwrite-secret |

## 8. Проверка корректности доступов

Команда:

```bash
bash k8s/test-rbac.sh
```

Результаты:

| Проверка | Ожидаемый результат | Фактический результат |
|---|---|---|
| model-serving-sa can get model-serving-secret | yes |  |
| model-serving-sa can get mlflow-db-secret | no |  |
| ci-cd-deployer-sa can create deployments | yes |  |
| ci-cd-deployer-sa can delete secrets | no |  |
| data-scientist-sa can get pods/log | yes |  |
| data-scientist-sa can get secrets | no |  |

## 9. Анализ безопасности

Ответы:

1. Какие секреты используются в ML-платформе?
2. Где они хранятся?
3. Кто имеет к ним доступ?
4. Какие права были ограничены?
5. Какие риски остаются?
6. Что произойдёт при утечке `VAULT_TOKEN`?
7. Почему опасно выдавать `cluster-admin` для CI/CD?
8. Какие меры можно применить для повышения безопасности?

Возможные улучшения:

- GitHub OIDC вместо статического Vault token;
- Vault audit logging;
- короткий TTL для Vault token;
- dynamic secrets;
- ротация секретов;
- External Secrets Operator;
- Sealed Secrets;
- разделение dev/stage/prod;
- protected environments;
- branch protection rules.

## 10. Выводы

В работе реализовано:

- централизованное хранение секретов в Vault;
- разграничение доступов через Vault policies;
- использование GitHub Secrets в CI/CD;
- GitHub Actions workflow без вывода секретов в логи;
- Kubernetes RBAC;
- проверка положительных и отрицательных сценариев доступа;
- анализ рисков и меры повышения безопасности.
'''.strip() + "\n", encoding="utf-8")

print("REPORT.md создан.")

## 9. Финальная проверка файлов

In [ ]:
# Выводим список созданных файлов.

for path in sorted(Path(".").rglob("*")):
    if path.is_file() and ".ipynb_checkpoints" not in str(path):
        print(path)

## 10. Рекомендуемый порядок выполнения

1. Установить Vault CLI, kubectl, Docker.
2. Запустить Vault в dev-режиме.
3. Выполнить `vault/setup-vault.sh`.
4. Проверить Vault-доступы через `vault/test-vault-access.sh`.
5. Создать GitHub Secrets.
6. Проверить GitHub Actions workflow.
7. Применить Kubernetes RBAC manifests.
8. Проверить доступы через `k8s/test-rbac.sh`.
9. Заполнить `REPORT.md`.